# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

Runs offline on the bundled slice. Builds the honest feature vector and then **attacks its own features** with deliberate leaks.

## 1. Build the feature vector

Unit: **one row = one already-visible page**. All features come from the prior-30 window or static
metadata; the label comes from the later-30 window.

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


In [2]:
print("feature matrix:", X.shape)
print("numeric:", len(NUM_FEATURES), "->", NUM_FEATURES)
print("categorical:", len(CAT_FEATURES), "->", CAT_FEATURES)

feature matrix: (18010, 43)
numeric: 18 -> ['log_impr_prev', 'log_clk_prev', 'log_sess_prev', 'ctr_prev', 'clk_per_sess_prev', 'ctr_prev_z', 'impr_prev_share', 'clk_prev_share', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'has_keyword', 'has_clicks_prev', 'has_sessions_prev']
categorical: 6 -> ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| feature | meaning | missing handling | available before the outcome window? |
|---|---|---|---|
| `log_impr_prev` | log(1+prior-30 impressions) | none | yes |
| `log_clk_prev`, `log_sess_prev` | prior-30 clicks / sessions | none | yes |
| `ctr_prev` | prior-30 clicks/impressions | 0 rows kept | yes |
| `clk_per_sess_prev` | prior-30 clicks per session | NaN→0 | yes |
| `ctr_prev_z` | prior-30 CTR vs the page's own client | NaN→0 (single-page client) | yes |
| `impr_prev_share`, `clk_prev_share` | share of the client's prior-30 totals | NaN→0 | yes |
| `search_volume`, `competition`, `cpc` | keyword context | NaN→0 + `has_keyword` flag | yes (static) |
| `word_count`, `char_count` | depth | NaN→0 (not a blind fill: a `has_*`-style flag column is unused here) | yes (static) |
| `content_age_days`, `days_since_last_update` | age / freshness | none | yes |
| categorical tiers | bucketed descriptors | `"unknown"` | yes |

The missingness follows `content_type` (see `w03_data_contract.ipynb` §3), so numeric fills are paired
with presence flags rather than being trusted blindly.

## 3. The leakage hunt

The attack: add one seductive column at a time to the *same* split and watch the AUC jump. `momentum =
last30/prev30` is literally the label's ratio; `trend_pct` is the label's source; `impressions_last_30d`
is the outcome window.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit


def cv_auc(frame, y_true, grp, seed=42):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed).split(frame, y_true, grp))
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=15, max_features="sqrt",
                                random_state=seed, n_jobs=-1).fit(frame.iloc[tr], y_true[tr])
    return round(float(roc_auc_score(y_true[te], rf.predict_proba(frame.iloc[te])[:, 1])), 3)


honest_auc = cv_auc(X, y, groups)

leak1 = X.copy()
leak1["momentum"] = (raw.loc[pool.index, "impressions_last_30d"]
                     / raw.loc[pool.index, "impressions_prev_30d"].clip(lower=1)).replace([np.inf, -np.inf], 0).fillna(0).values
leak1_auc = cv_auc(leak1, y, groups)

leak2 = X.copy()
leak2["trend_pct"] = pd.to_numeric(raw.loc[pool.index, "trend_pct"], errors="coerce").fillna(0).values
leak2_auc = cv_auc(leak2, y, groups)

leak3 = X.copy()
leak3["impressions_last_30d"] = raw.loc[pool.index, "impressions_last_30d"].values
leak3_auc = cv_auc(leak3, y, groups)

print(f"honest feature set (grouped by client)      AUC = {honest_auc}")
print(f"+ momentum (last30/prev30)                  AUC = {leak1_auc}   <- the label in disguise")
print(f"+ raw trend_pct                             AUC = {leak2_auc}")
print(f"+ impressions_last_30d (outcome window)     AUC = {leak3_auc}")
print()
print("VERDICT: any single column from the label's comparison window lifts AUC toward 1.0 on the")
print("SAME split. That is leakage, not skill. Rule: on this snapshot no feature may be derived")
print("from the last-30 window or from the 90-day totals that contain it.")

honest feature set (grouped by client)      AUC = 0.579
+ momentum (last30/prev30)                  AUC = 1.0   <- the label in disguise
+ raw trend_pct                             AUC = 1.0
+ impressions_last_30d (outcome window)     AUC = 0.848

VERDICT: any single column from the label's comparison window lifts AUC toward 1.0 on the
SAME split. That is leakage, not skill. Rule: on this snapshot no feature may be derived
from the last-30 window or from the 90-day totals that contain it.


## 4. What I excluded and why

| excluded column(s) | why |
|---|---|
| `trend_direction`, `trend_pct` | they define the label |
| `*_last_30d` | the label's outcome window |
| all `*_90d` totals and their rates (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, coverage days) | a 90-day window contains the outcome window |
| `impression_tier`, `position_tier` | bucketed from the 90-day columns |
| `provider_used`, `model_used` | production/LLM metadata, not a page signal |
| `content_id`, `client_id` | identifiers → grouping only; no names, URLs, or queries exist in the file |

In [4]:
metrics = {
    "task": "ml-05",
    "data": "data/raw/content_refresh_anonymized.csv",
    "rows": int(len(raw)),
    "pool_rows": int(len(pool)),
    "feature_matrix_shape": list(X.shape),
    "label": "declined_next_30d (later-30 impressions < 80% of prior-30)",
    "honest_auc": honest_auc,
    "leak_momentum_auc": leak1_auc,
    "leak_trend_pct_auc": leak2_auc,
    "leak_outcome_window_auc": leak3_auc,
    "decision": "no feature may be derived from the outcome window or from 90-day totals",
}
(OUT / "w03_feature_leakage_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("receipt written: work/outputs/w03_feature_leakage_metrics.json")

receipt written: work/outputs/w03_feature_leakage_metrics.json


### Self-check

- [x] Feature vector built by code, not described
- [x] Every feature's availability checked against the outcome window
- [x] Leakage shown as a number (honest vs three deliberate leaks)
- [x] Excluded list written with a reason each
- [x] No client names, URLs, or private queries anywhere
- [x] Committed under `work/notebooks/`